In [1]:
import torch
from geister_game import GeisterGame
from train import run_geister_cnn_training, run_geister_cqcnn_training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
# CNNモデルの学習・評価
run_geister_cnn_training(episodes=300)


--- Training CNN Agent for Geister ---


NameError: name 'PLAYER_A_ID' is not defined

In [17]:
# CQCNNモデルの学習・評価
run_geister_cqcnn_training(episodes=1000, n_qbits=6, cnn_out_feat=6)


--- Training CQCNN Agent (Qubits: 6, CNN->QNN Feat: 6) ---
Initialized QNN device: lightning.qubit with 6 qubits.
Game ended: Draw by turn limit.
Agent A CQCNN model saved to ./models_geister_agentA/agentA_cqcnn_eps20.pth
Agent B CQCNN model saved to ./models_geister_agentB/agentB_cqcnn_eps20.pth
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Agent A CQCNN model saved to ./models_geister_agentA/agentA_cqcnn_eps40.pth
Agent B CQCNN model saved to ./models_geister_agentB/agentB_cqcnn_eps40.pth
Game ended: Draw by turn limit.
Agent A CQCNN model saved to ./models_geister_agentA/agentA_cqcnn_eps60.pth
Agent B CQCNN model saved to ./models_geister_agentB/agentB_cqcnn_eps60.pth
Game ended: Draw by turn limit.
Agent A CQCNN model saved to ./models_geister_agentA/agentA_cqcnn_eps80.pth
Agent B CQCNN model saved to ./models_geister_agentB/agentB_cqcnn_eps80.pth
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by tu

In [2]:
import json
import torch
import pennylane as qml
from model_cqcnn import CNN_QNN_CNN_Geister

# 設定ファイルの読み込み
with open("models_geister_agentA/agentA_cqcnn_config.json", "r") as f:
    config = json.load(f)
# 正しい PennyLane デバイスを構築
dev = qml.device(config.get("dev_type", "default.qubit"), wires=config["n_qubits_qnn"])


# モデルの構築
model = CNN_QNN_CNN_Geister(
    dev=dev,  # ← 修正ポイント
    n_qubits_qnn=config["n_qubits_qnn"],
    exp_or_prob=config["exp_or_prob"],
    embedding_type=config["embedding_type"],
    ansatz_type=config["ansatz_type"],
    feature_map_reps=config["feature_map_reps"],
    ansatz_reps=config["ansatz_reps"],
    input_channels_cnn=config["input_channels_cnn"],
    board_size_cnn=config["board_size_cnn"],
    cnn_fc_out_features=config["cnn_fc_out_features"],
    qnn_fc_out_features=config["qnn_fc_out_features"]
)

# 重みの読み込み
model.load_state_dict(torch.load("models_geister_agentA/agentA_cqcnn_eps500.pth", map_location=device))
model.eval()  # 評価モードにする

CNN_QNN_CNN_Geister(
  (cnn_feature_extractor): Sequential(
    (0): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Flatten(start_dim=1, end_dim=-1)
  )
  (fc_to_qnn): Linear(in_features=1152, out_features=6, bias=True)
  (fc_from_qnn): Linear(in_features=6, out_features=36, bias=True)
)

In [15]:
# ゲームインスタンスとエージェントを作成
from train import Env_Geister
from train import AgentFactory
game = GeisterGame(board_size=6)
# 設定をファイルから読み込む
with open('models_geister_agentA/agentA_cqcnn_config.json', 'r') as f:
    configA = json.load(f)

with open('models_geister_agentA/agentA_cqcnn_config.json', 'r') as f:
    configB = json.load(f)
agentA = AgentFactory.create_cqc_agent("A", game, configA, 'models_geister_agentA/agentA_cqcnn_eps100.pth')
agentB = AgentFactory.create_cqc_agent("B", game, configB, 'models_geister_agentA/agentA_cqcnn_eps3000.pth')

# 環境インスタンス作成
env = Env_Geister(agentA, agentB, game)

# 対戦実行
winner, moves_log, final_board = env.play_one_game_with_log()
print("Winner:", winner)
print("Moves Log:", moves_log)

Winner: B
Moves Log: [{'turn': 1, 'player': 'A', 'action': ((4, 2), (3, 2))}, {'turn': 2, 'player': 'B', 'action': ((0, 1), (0, 0))}, {'turn': 3, 'player': 'A', 'action': ((3, 2), (2, 2))}, {'turn': 4, 'player': 'B', 'action': ((0, 0), (1, 0))}, {'turn': 5, 'player': 'A', 'action': ((4, 1), (4, 0))}, {'turn': 6, 'player': 'B', 'action': ((0, 2), (0, 1))}, {'turn': 7, 'player': 'A', 'action': ((4, 4), (3, 4))}, {'turn': 8, 'player': 'B', 'action': ((1, 0), (2, 0))}, {'turn': 9, 'player': 'A', 'action': ((2, 2), (2, 1))}, {'turn': 10, 'player': 'B', 'action': ((1, 4), (1, 5))}, {'turn': 11, 'player': 'A', 'action': ((5, 1), (4, 1))}, {'turn': 12, 'player': 'B', 'action': ((1, 5), (0, 5))}, {'turn': 13, 'player': 'A', 'action': ((3, 4), (3, 3))}, {'turn': 14, 'player': 'B', 'action': ((0, 5), (1, 5))}, {'turn': 15, 'player': 'A', 'action': ((3, 3), (3, 2))}, {'turn': 16, 'player': 'B', 'action': ((0, 1), (0, 0))}, {'turn': 17, 'player': 'A', 'action': ((4, 3), (3, 3))}, {'turn': 18, 'play

In [11]:
print("\n--- Evaluating Trained CQCNN Agent A vs Random ---")
agentA.eval_mode_on()
agentA.epsilon = 0.0 # 評価時はランダム性なし
agentB.eval_mode_on()
agentB.epsilon = 0.0 # 評価時はランダム性なし
eval_env_Q_vs_env_Q = env
eval_env_Q_vs_env_Q.start_training(episodes=100, visualize_interval=0, train_agents=False)


--- Evaluating Trained CQCNN Agent A vs Random ---
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by turn limit.
Game ended: Draw by 